**Mini Swing Trading** está muy bien planteada para alguien que busca rentabilidad sin el estrés y la volatilidad extrema del Day Trading. Filtrar por activos de alta liquidez ($AAPL, $MSFT, $NVDA, $SPY$) y buscar compras en pullbacks (retrocesos) es un enfoque clásico y robusto.

### Optimización de la Estrategia para Alpaca

Para que esta estrategia funcione de forma óptima bajo tu presupuesto sugerido (€200–500) y las reglas que has puesto, debemos ajustar un par de detalles técnicos:

- Capital y Acciones Fraccionadas: Con un capital de 200–500 €, comprar acciones enteras de NVDA o MSFT (que suelen cotizar a cientos de dólares) limitaría tu diversificación. La API de Alpaca permite acciones fraccionadas (fractional shares), lo cual es perfecto para ti. Podrás comprar, por ejemplo, 50 € de NVDA sin importar su precio por acción.

- La Media Móvil de 20 días (SMA 20): En un pullback, el precio idealmente debería rebotar cerca o justo por encima de la SMA 20. Comprar cuando el precio cruza hacia abajo la SMA 20 (pero el RSI indica sobreventa temporal) suele ser el punto de gatillo.

- Gestión del Riesgo (Stop Loss): Para mantener el "bajo estrés", es vital definir un Stop Loss. Si buscas un 3–6% de ganancia, tu Stop Loss debería estar en torno al 1.5% o 3% (manteniendo un ratio riesgo/beneficio de al menos 1:2).

In [1]:
!pip install alpaca-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 3.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest, LimitOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from datetime import datetime, timedelta
from google.colab import userdata

In [3]:
API_KEY = userdata.get('ALPACA_KEY')
SECRET_KEY = userdata.get('ALPACA_SECRET')

trading_client = TradingClient(API_KEY, SECRET_KEY, paper=True)
data_client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

In [4]:
# Configuración de la estrategia
ACTIVOS = ["AAPL", "MSFT", "NVDA", "SPY"]
CAPITAL_POR_ACTIVO = 100.0  # Euros/Dólares a invertir por señal

In [7]:
for simbolo in ACTIVOS:
    # 2. Obtener datos históricos de los últimos 40 días para calcular los indicadores
    request_params = StockBarsRequest(
        symbol_or_symbols=simbolo,
        timeframe=TimeFrame.Day,
        start=datetime.now() - timedelta(days=60)
    )
    bars = data_client.get_stock_bars(request_params).df

    # Calcular SMA 20 y RSI de 14 días (fórmula simplificada en pandas)
    bars['SMA_20'] = bars['close'].rolling(window=20).mean()

    delta = bars['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    bars['RSI_14'] = 100 - (100 / (1 + rs))

    # Últimos valores calculados (los de hoy)
    precio_actual = bars['close'].iloc[-1]
    sma_actual = bars['SMA_20'].iloc[-1]
    rsi_actual = bars['RSI_14'].iloc[-1]

    # 3. Lógica de Compra (Condiciones)
    # Buscamos que el precio esté cerca o tocando la SMA 20 y el RSI sea bajo
    if precio_actual <= (sma_actual * 1.01) and rsi_actual < 40:
        # Verificar si ya tenemos posición en este activo
        try:
            posicion = trading_client.get_open_position(simbolo)
            print(f"Ya tienes una posición en {simbolo}. Saltando...")
        except:
            # No hay posición, procedemos a comprar usando monto en dólares (fraccional)
            order_data = MarketOrderRequest(
                symbol=simbolo,
                notional=CAPITAL_POR_ACTIVO, # Compra basada en dinero, no en número de acciones
                side=OrderSide.BUY,
                time_in_force=TimeInForce.DAY
            )
            trading_client.submit_order(order_data=order_data)
            print(f"🚀 Orden de COMPRA enviada para {simbolo}. RSI: {rsi_actual:.2f}")

            # 4. Lógica de Venta (Si ya estás dentro)
    try:
        posicion = trading_client.get_open_position(simbolo)
        ganancia_actual = float(posicion.unrealized_plpc) * 100 # Porcentaje de ganancia

        # Vender si el RSI > 60 o si alcanzamos el objetivo del 3-6%
        if rsi_actual > 60 or ganancia_actual >= 4.5:
            trading_client.close_position(simbolo)
            print(f"💰 Orden de VENTA enviada para {simbolo}. Ganancia: {ganancia_actual:.2f}%")
    except:
        pass # No hay posición que vender

💰 Orden de VENTA enviada para AAPL. Ganancia: -0.60%


### Consejos para llevarla a cabo con éxito

- Usa Alpaca Paper Trading primero: Alpaca te da un entorno de simulación idéntico al real con dinero ficticio. Prueba este algoritmo o tu operativa manual ahí durante al menos 2 semanas para ver cómo reacciona en los marcos de tiempo de 2 a 10 días.

- Cuidado con el "Wash Sale" (Regla del Impuesto): Si operas desde EE. UU. o ciertos países con tratados, vender con pérdidas y volver a comprar el mismo activo antes de 30 días puede tener implicaciones fiscales. Al ser Swing de 2-10 días, es poco probable que te afecte mucho, pero tenlo en mente.

- El horario del mercado: Como solo te tomará 15 minutos, el mejor momento para revisar los indicadores y ejecutar las órdenes es 30 minutos antes del cierre del mercado de Nueva York (alrededor de las 21:30 hora de España Central). Así te aseguras de que la vela diaria está casi consolidada y evitas las falsas alarmas de la apertura.